In [0]:
%run "/Users/f3n2x_2005@yahoo.com/resale/common01"

In [0]:
#configure file location path
file_location = DATA_FOLDER + "input/*"

In [0]:
#load source file into dataframe
from pyspark.sql.types import StructType, StructField,StringType,IntegerType,DoubleType,DateType

schema1 = StructType([
    StructField("month",StringType(),True),
    StructField("town",StringType(),True),
    StructField("flat_type",StringType(),True),
    StructField("block",StringType(),True),
    StructField("street_name",StringType(),True),
    StructField("storey_range",StringType(),True),
    StructField("floor_area_sqm",DoubleType(),True),
    StructField("flat_model",StringType(),True),
    StructField("lease_commence_date",IntegerType(),True),
    StructField("remaining_lease",DoubleType(),True),
    StructField("resale_price_2015",DoubleType(),True)    
])

df_init = spark.read.option("delimiter",",").option("header","True").schema(schema1).csv(file_location)

In [0]:
# data profiling
df_init.describe().display()



summary,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price_2015
count,694254,694254,694254,694254,694254,694254,694254,694254,694254,459007,272400
mean,null,null,null,351.1953309478625,null,null,96.61537823332661,null,1991.1642309586982,278976.05194715987,519073.9295011013
stddev,null,null,null,258.0948249906707,null,null,24.95283131662255,null,11.99053434113037,148017.46929924985,187113.83491357198
min,2000-01,ANG MO KIO,1 ROOM,1,ADMIRALTY DR,01 TO 03,28.0,2-room,1966,48.0,140000.0
max,2026-07,YISHUN,MULTI-GENERATION,9B,ZION RD,49 TO 51,366.7,Type S2,2022,1088888.0,1728000.0


In [0]:
# data profiling null value 
from pyspark.sql.functions import col, sum

# for c in df_init.columns:
#     print(c, df_init.filter(col(c).isNull()).count())
df_init.select(*(sum(col(c).isNull().cast("int")).alias(c) for c in df_init.columns)).show()


+-----+----+---------+-----+-----------+------------+--------------+----------+-------------------+---------------+-----------------+
|month|town|flat_type|block|street_name|storey_range|floor_area_sqm|flat_model|lease_commence_date|remaining_lease|resale_price_2015|
+-----+----+---------+-----+-----------+------------+--------------+----------+-------------------+---------------+-----------------+
|    0|   0|        0|    0|          0|           0|             0|         0|                  0|         235247|           421854|
+-----+----+---------+-----+-----------+------------+--------------+----------+-------------------+---------------+-----------------+



In [0]:
# 2015 file has extra column (remaining_lease), 
# 2012 file has no remaining_lease, thus resale_price will be reside in (remaining_lease) and (resale_price_t) will be null
# to handle both files in 1 single notebook, create a new column(resale_price) apply a condition to validate if 
# resale_price_t is null(2012 file), then resale_price = remaining_lease, else resale_price = resale_price_t  
# verify if there's resale_price < 1000 
from pyspark.sql.functions import when
df_resale = df_init.withColumn("resale_price", when( df_init["resale_price_2015"].isNull(), df_init["remaining_lease"] )\
                .otherwise(df_init["resale_price_2015"]) )\
                .select(df_init["month"],df_init["town"],df_init["flat_type"],df_init["block"],\
                    df_init["street_name"],df_init["storey_range"],df_init["floor_area_sqm"],df_init["flat_model"],\
                    df_init["lease_commence_date"],df_init["resale_price"])
df_resale. filter(df_resale["resale_price"] < 1000).display()

month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price


In [0]:
# save raw file
file_location = DATA_FOLDER + '/out/raw'
df_resale.coalesce(1).write.format("csv").mode("append").option("header","true").save(file_location)

In [0]:
df_resale.describe().display()

summary,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price
count,694254,694254,694254,694254,694254,694254,694254,694254,694254,694254
mean,null,null,null,351.1953309478625,null,null,96.61537823332661,null,1991.1642309586982,388107.1667980451
stddev,null,null,null,258.0948249906707,null,null,24.95283131662255,null,11.99053434113037,186463.83380871403
min,2000-01,ANG MO KIO,1 ROOM,1,ADMIRALTY DR,01 TO 03,28.0,2-room,1966,28000.0
max,2026-07,YISHUN,MULTI-GENERATION,9B,ZION RD,49 TO 51,366.7,Type S2,2022,1728000.0


In [0]:
# to filter only 2012 Jan to 2016 Dec data extracted
df_2012 = df_resale.filter("month >= '2012-01' and month <= '2016-12'")


In [0]:
# demonstrate validation function to drop any null value or 0
# df_valid_month = nullValid(df_2012, "flat_type")
# df_valid_town = nullValid(df_valid_month, "town")
# df_great0_floor = great0(df_valid_town,"floor_area_sqm")
# df_great0_resale = great0(df_great0_floor,"resale_price")

In [0]:
# since there is no information other than lease_commence_date (year), I'll assume lease start date is 01 JAN of commence year
# compute lease end date and remaining lease year and months against 
from pyspark.sql.functions import datediff, to_date,col,lit,concat,months_between,current_date
df_lease_end = df_2012.withColumn("Lease_End_DT",to_date(concat(lit("01-01-"),df_2012['lease_commence_date'] + 99),"dd-MM-yyyy") )
df_year_rem = df_lease_end.withColumn("Remain_Year",(months_between(df_lease_end["Lease_End_DT"],current_date())/12).cast("int")) \
    .withColumn("Remain_Month",(months_between(df_lease_end["Lease_End_DT"],current_date())%12).cast("int"))


In [0]:
# data validation duplicate record, but this is for multiple floor, it may be valid duplicate
df_year_rem.groupby("month","town","flat_type","block","street_name","storey_range",\
    "floor_area_sqm","flat_model","lease_commence_date",\
    "resale_price","Remain_Year","Remain_Month").count()\
    .filter("count > 1").display()

month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,Remain_Year,Remain_Month,count
2013-08,BEDOK,3 ROOM,77,BEDOK NTH RD,01 TO 03,60.0,Improved,1986,300000.0,58,5,2
2014-05,ANG MO KIO,3 ROOM,556,ANG MO KIO AVE 10,01 TO 03,68.0,New Generation,1980,300000.0,52,5,2
2014-05,JURONG WEST,4 ROOM,909,JURONG WEST ST 91,04 TO 06,104.0,Model A,1989,380000.0,61,5,2
2014-04,JURONG WEST,5 ROOM,653C,JURONG WEST ST 61,04 TO 06,110.0,Improved,2002,474500.0,74,5,2
2014-05,KALLANG/WHAMPOA,3 ROOM,105,TOWNER RD,07 TO 09,74.0,Model A,1984,440000.0,56,5,2
2014-05,YISHUN,3 ROOM,264,YISHUN ST 22,01 TO 03,64.0,Simplified,1986,285000.0,58,5,2
2014-01,SEMBAWANG,4 ROOM,486,ADMIRALTY LINK,10 TO 12,90.0,Model A,2004,420000.0,76,5,2
2014-04,QUEENSTOWN,4 ROOM,43,TANGLIN HALT RD,04 TO 06,129.0,Adjoined flat,1971,605000.0,43,5,2
2014-04,SERANGOON,4 ROOM,144,SERANGOON NTH AVE 1,01 TO 03,103.0,Model A,1987,408000.0,59,5,2
2014-05,ANG MO KIO,3 ROOM,608,ANG MO KIO AVE 5,01 TO 03,81.0,New Generation,1980,380000.0,52,5,2


In [0]:
# data validation MOP < 5 years
df_year_rem.filter(" lease_commence_date +5 >  cast(substr(month,1,4) as int )").display()

In [0]:
# to filter only highest resale price if others remaining column are same 
from pyspark.sql.window import Window
from pyspark.sql.functions import rank,desc

winSpec = Window.partitionBy(["month","town","flat_type","block","street_name","storey_range",\
                              "floor_area_sqm","flat_model","lease_commence_date"])\
.orderBy(desc("resale_price"))

df_resale = df_year_rem.select("*").withColumn("rank",rank().over(winSpec))


In [0]:
df_resale_1 = df_resale.filter("rank = 1")
df_resale_fail = df_resale.filter("rank > 1")


In [0]:
df_resale_fail.display()

month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,Lease_End_DT,Remain_Year,Remain_Month,rank
2012-01,ANG MO KIO,3 ROOM,256,ANG MO KIO AVE 4,10 TO 12,73.0,New Generation,1977,340000.0,2076-01-01,49,5,2
2012-01,ANG MO KIO,3 ROOM,506,ANG MO KIO AVE 8,07 TO 09,68.0,New Generation,1980,334000.0,2079-01-01,52,5,2
2012-01,ANG MO KIO,3 ROOM,633,ANG MO KIO AVE 6,10 TO 12,67.0,New Generation,1985,350000.0,2084-01-01,57,5,2
2012-01,BUKIT MERAH,3 ROOM,37,JLN RUMAH TINGGI,04 TO 06,53.0,Standard,1969,303000.0,2068-01-01,41,5,2
2012-01,BUKIT MERAH,3 ROOM,37,JLN RUMAH TINGGI,04 TO 06,53.0,Standard,1969,290000.0,2068-01-01,41,5,3
2012-01,PASIR RIS,4 ROOM,626,PASIR RIS DR 3,01 TO 03,104.0,Model A,1995,413000.0,2094-01-01,67,5,2
2012-01,SENGKANG,EXECUTIVE,202B,SENGKANG EAST RD,13 TO 15,130.0,Apartment,2001,630000.0,2100-01-01,73,5,2
2012-02,JURONG WEST,3 ROOM,516,JURONG WEST ST 52,04 TO 06,74.0,Model A,1984,380000.0,2083-01-01,56,5,2
2012-02,SENGKANG,5 ROOM,311A,ANCHORVALE LANE,04 TO 06,111.0,Premium Apartment,2002,475000.0,2101-01-01,74,5,2
2012-02,TAMPINES,3 ROOM,884,TAMPINES ST 83,04 TO 06,74.0,Model A,1988,350000.0,2087-01-01,60,5,2


In [0]:
file_location = DATA_FOLDER + '/out/fail'
df_resale_fail.coalesce(1).write.format("csv").mode("append").option("header","true").save(file_location)

In [0]:
file_location = DATA_FOLDER + '/out/valid'
df_resale_1.coalesce(1).write.format("csv").mode("append").option("header","true").save(file_location)